[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apartsin/DLCourseHIT/blob/main/notebooks/week09.ipynb)

# Week 9: Convolutional Networks II
**Introduction to Deep Learning (HIT)** &middot; Part III: Architectures & Representation Learning

Batch and layer normalization; residual connections; modern CNN design.

**Instructor practice notebook.** Run these live demonstrations during the 2-hour practice lesson. The student homework is the weekly lab.

## Goals for the practice lesson

- Add normalization and residual connections.
- Understand why these help deeper networks train.
- Measure the effect of each with an ablation.

## Setup
Run this first. On Colab, set Runtime > Change runtime type > GPU for the later weeks.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(0)
print('PyTorch', torch.__version__, '| device:', device)

## Demonstration 1
Add batch normalization and residual blocks to the CNN.

In [ ]:
# A residual block with batch normalization (keeps the input shape)
class ResBlock(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.c1 = nn.Conv2d(c, c, 3, padding=1); self.bn1 = nn.BatchNorm2d(c)
        self.c2 = nn.Conv2d(c, c, 3, padding=1); self.bn2 = nn.BatchNorm2d(c)
    def forward(self, x):
        h = F.relu(self.bn1(self.c1(x)))
        h = self.bn2(self.c2(h))
        return F.relu(h + x)               # the skip (residual) connection
print("ResBlock keeps shape:", tuple(ResBlock(16)(torch.randn(2, 16, 8, 8)).shape))

## Demonstration 2
Ablate normalization and residuals live and compare the training curves.

In [ ]:
# Batch norm helps a deep network train (ablation)
torch.manual_seed(0)
X = torch.randn(256, 16); y = torch.randint(0, 2, (256,))
def deepnet(bn):
    layers = []
    for _ in range(6):
        layers.append(nn.Linear(16, 16))
        if bn: layers.append(nn.BatchNorm1d(16))
        layers.append(nn.ReLU())
    return nn.Sequential(*layers, nn.Linear(16, 2))
for bn in [False, True]:
    m = deepnet(bn); o = torch.optim.SGD(m.parameters(), 0.1); f = nn.CrossEntropyLoss(); h = []
    for _ in range(60):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
    plt.plot(h, label="with BN" if bn else "no BN")
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.title("Normalization ablation"); plt.show()

## Demonstration 3
Show a deeper network failing without residuals and training with them.

In [ ]:
# Residual connections let a deep network keep training
class Deep(nn.Module):
    def __init__(self, res):
        super().__init__(); self.res = res
        self.blocks = nn.ModuleList([nn.Linear(16, 16) for _ in range(12)])
        self.head = nn.Linear(16, 2)
    def forward(self, x):
        for b in self.blocks:
            h = F.relu(b(x)); x = h + x if self.res else h
        return self.head(x)
for res in [False, True]:
    m = Deep(res); o = torch.optim.SGD(m.parameters(), 0.05); f = nn.CrossEntropyLoss(); h = []
    for _ in range(80):
        o.zero_grad(); l = f(m(X), y); l.backward(); o.step(); h.append(l.item())
    plt.plot(h, label="residual" if res else "plain")
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.title("Residual vs plain (12 layers)"); plt.show()

---
Student materials for this week: the lab handout (`labs/week09.html`) and the curated references (`references/week09.html`) in the course site.